In [0]:
# Gold Layer: customer_360 + journey_events

#Builds the **Gold layer** — analytics-ready tables joined and aggregated from Silver.

### Tables produced
#1. **`customer_360`** — one row per customer with all behavioral, financial, and experience metrics
#2. **`journey_events`** — chronological event log per customer (powers journey mapping in Week 3)

### Why Gold matters
#Silver is trustworthy but normalized — analysts would have to join 4 tables every time.
#Gold pre-joins and aggregates for fast, business-ready consumption.
#"""


In [0]:
CATALOG = "cx"
SCHEMA_SILVER = "cx_silver"
SCHEMA_GOLD = "cx_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
print(f"Schema ready: {CATALOG}.{SCHEMA_GOLD}")

Schema ready: cx.cx_gold


In [0]:
customers = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers")
tickets = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.service_interactions")
usage = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.product_usage")
surveys = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.surveys")

print(f"customers: {customers.count():,}")
print(f"tickets: {tickets.count():,}")
print(f"usage: {usage.count():,}")
print(f"surveys: {surveys.count():,}")


customers: 50,000
tickets: 352,550
usage: 1,160,000
surveys: 29,745


In [0]:
from pyspark.sql.functions import (
    col, count, sum as sum_, avg, max as max_, min as min_,
    when, datediff, current_date, expr, countDistinct, round as round_
)


In [0]:
#Ticket aggregates per customer

In [0]:
ticket_agg = (
    tickets
    .filter(col("dq_flag_orphan") == 0)  # exclude orphan tickets
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_tickets"),
        sum_(when(col("priority") == "Critical", 1).otherwise(0)).alias("critical_tickets"),
        sum_(col("escalated_flag")).alias("escalated_tickets"),
        round_(avg("first_response_minutes"), 2).alias("avg_first_response_min"),
        round_(avg("resolution_hours"), 2).alias("avg_resolution_hours"),
        round_(avg("csat_post_ticket"), 2).alias("avg_post_ticket_csat"),
        max_("ticket_open_date").alias("last_ticket_date"),
    )
)
print(f"ticket_agg: {ticket_agg.count():,} customers with tickets")


ticket_agg: 48,746 customers with tickets


In [0]:
# Usage aggregates
# Last 12 months of usage as "current state"

In [0]:
usage_recent = usage.filter(col("snapshot_month") >= expr("add_months(current_date(), -12)"))

usage_agg = (
    usage_recent
    .groupBy("customer_id")
    .agg(
        round_(avg("test_volume"), 0).alias("avg_monthly_test_volume"),
        round_(avg("uptime_pct"), 4).alias("avg_uptime_pct"),
        round_(avg("error_events_count"), 2).alias("avg_monthly_errors"),
        max_("active_instruments").alias("active_instruments"),
        countDistinct("software_version").alias("software_versions_used"),
    )
)
print(f"usage_agg: {usage_agg.count():,} customers with telemetry")

usage_agg: 40,000 customers with telemetry


In [0]:
# Survey aggregates per customer (most recent NPS)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Most recent NPS response per customer
window_recent_survey = Window.partitionBy("customer_id").orderBy(col("survey_date").desc())

most_recent_survey = (
    surveys
    .filter(col("nps_score").isNotNull())
    .withColumn("rn", row_number().over(window_recent_survey))
    .filter(col("rn") == 1)
    .select(
        "customer_id",
        col("nps_score").alias("latest_nps"),
        col("nps_bucket").alias("latest_nps_bucket"),
        col("csat_score").alias("latest_csat"),
        col("ces_score").alias("latest_ces"),
        col("survey_date").alias("latest_survey_date"),
    )
)

# Lifetime survey volume
survey_volume = (
    surveys
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_survey_responses"),
        round_(avg("nps_score"), 2).alias("avg_nps_lifetime"),
    )
)

survey_agg = most_recent_survey.join(survey_volume, "customer_id", "left")
print(f"survey_agg: {survey_agg.count():,} customers with surveys")

survey_agg: 14,958 customers with surveys


In [0]:
#Build customer_360 — join everything

In [0]:
customer_360 = (
    customers
    .join(ticket_agg, "customer_id", "left")
    .join(usage_agg, "customer_id", "left")
    .join(survey_agg, "customer_id", "left")
    # Derived features
    .withColumn("tenure_days", datediff(current_date(), col("contract_start_date")))
    .withColumn("tenure_years", round_(col("tenure_days") / 365.25, 2))
    .withColumn(
        "tickets_per_year",
        round_(col("total_tickets") / when(col("tenure_years") > 0, col("tenure_years")).otherwise(1), 2)
    )
    .withColumn(
        "has_recent_critical",
        when(col("critical_tickets") > 0, 1).otherwise(0)
    )
    .withColumn(
        "engagement_score",
        round_(
            (when(col("avg_monthly_test_volume").isNotNull(), 1).otherwise(0) +
             when(col("total_survey_responses") > 0, 1).otherwise(0) +
             when(col("active_instruments") > 1, 1).otherwise(0)) / 3.0,
            2
        )
    )
    # Fill nulls for customers without tickets/usage/surveys
    .fillna({
        "total_tickets": 0,
        "critical_tickets": 0,
        "escalated_tickets": 0,
        "total_survey_responses": 0,
        "active_instruments": 0,
    })
)

customer_360.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.customer_360"
)
print(f"\n✅ customer_360: {customer_360.count():,} rows")



✅ customer_360: 50,000 rows


In [0]:
## Build journey_events table
##Chronological log of every "event" per customer — used in Week 3 for journey mapping and funnel analysis.

In [0]:
from pyspark.sql.functions import lit

# Event 1: Contract start (Onboarding)
contract_events = customers.select(
    "customer_id",
    col("contract_start_date").alias("event_date"),
    lit("contract_start").alias("event_type"),
    lit("Onboarding").alias("journey_stage"),
    lit(None).cast("string").alias("event_detail"),
    lit(None).cast("double").alias("event_value"),
)

# Event 2: Support tickets
ticket_events = (
    tickets
    .filter(col("dq_flag_orphan") == 0)
    .select(
        "customer_id",
        col("ticket_open_date").cast("date").alias("event_date"),
        lit("support_ticket").alias("event_type"),
        when(col("priority").isin("Critical", "High"), lit("Friction"))
            .otherwise(lit("Support")).alias("journey_stage"),
        col("issue_category").alias("event_detail"),
        col("resolution_hours").alias("event_value"),
    )
)

# Event 3: Surveys
survey_events = surveys.select(
    "customer_id",
    col("survey_date").alias("event_date"),
    lit("survey_response").alias("event_type"),
    col("nps_bucket").alias("journey_stage"),
    lit("NPS_response").alias("event_detail"),
    col("nps_score").cast("double").alias("event_value"),
)

# Union all events
journey_events = contract_events.unionByName(ticket_events).unionByName(survey_events)
journey_events = journey_events.filter(col("event_date").isNotNull())

journey_events.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_GOLD}.journey_events"
)
print(f"✅ journey_events: {journey_events.count():,} rows")

✅ journey_events: 431,549 rows


In [0]:
%sql
--Sanity check 1: customer_360 quick stats
 SELECT
   COUNT(*) AS total_customers,
   COUNT(latest_nps) AS customers_with_nps,
   COUNT(avg_uptime_pct) AS customers_with_usage,
   ROUND(AVG(total_tickets), 1) AS avg_tickets_per_cust,
   ROUND(AVG(annual_recurring_revenue_usd), 0) AS avg_arr
 FROM cx.cx_gold.customer_360

total_customers,customers_with_nps,customers_with_usage,avg_tickets_per_cust,avg_arr
50000,14958,40000,7.0,66167.0


In [0]:
%sql
--Sanity check 2: NPS bucket by service tier
 SELECT service_tier, latest_nps_bucket, COUNT(*) AS n,
        ROUND(AVG(annual_recurring_revenue_usd), 0) AS avg_arr
 FROM cx.cx_gold.customer_360
 WHERE latest_nps_bucket IS NOT NULL
 GROUP BY service_tier, latest_nps_bucket
 ORDER BY service_tier, latest_nps_bucket

service_tier,latest_nps_bucket,n,avg_arr
Bronze,Detractor,3056,66043.0
Bronze,Passive,2428,64155.0
Bronze,Promoter,2726,65668.0
Gold,Detractor,811,67314.0
Gold,Passive,679,63714.0
Gold,Promoter,779,65955.0
Silver,Detractor,1703,66936.0
Silver,Passive,1343,67367.0
Silver,Promoter,1433,64077.0


In [0]:
%sql
--Sanity check 3: event type counts
 SELECT event_type, COUNT(*) AS n
 FROM cx.cx_gold.journey_events
 GROUP BY event_type
 ORDER BY n DESC

event_type,n
support_ticket,351804
contract_start,50000
survey_response,29745


In [0]:
%sql
SELECT 'original_customer_360' AS source, COUNT(*) AS n FROM cx.cx_gold.customer_360
UNION ALL
SELECT 'dlt_customer_360', COUNT(*) FROM cx.cx_gold.gold_customer_360_dlt
UNION ALL
SELECT 'original_journey_events', COUNT(*) FROM cx.cx_gold.journey_events
UNION ALL
SELECT 'dlt_journey_events', COUNT(*) FROM cx.cx_gold.gold_journey_events_dlt;

source,n
dlt_customer_360,50000
original_customer_360,50000
original_journey_events,431549
dlt_journey_events,431549
